# TRN Optogenetic Stimulation — Stimulus Inspector & Protocol Generator  (v6)

This notebook does three things:
1. **Generates** the pre-verified noise tables and trial sequence as C header files
   ready to drop into your Arduino sketch folder
2. **Visualises** every stimulus so you can verify waveforms and spectra
3. **Produces a session overview** — timeline, trial counts, estimated duration

**Stimulus set (220 trials, ~55 min):**
| Type | Stimuli | Reps | Total |
|------|---------|------|-------|
| Chirp | Fwd, Rev | 20 each | 40 |
| Noise | LP100, LP500 | 20 each | 40 |
| Sine | 2, 10, 25, 60, 100, 500 Hz | 20 each | 120 |
| Ramp | — | 20 | 20 |

> **Workflow:** Edit Section 1, run all cells, copy the two `.h` files into your
> Arduino sketch folder alongside `opto_stim_v6.ino`, then upload.


## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.signal import firwin, lfilter, welch, spectrogram
import os

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.titlesize': 11,
})
print('Imports OK')

## 1. Parameters

> **Edit only this cell.** Everything downstream derives from these values.
> Keep in sync with `USER PARAMETERS` in `opto_stim_v6.ino`.


In [ ]:
# ── DAC ──────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 10000       # Hz
STIM_DURATION = 5.0         # seconds
DAC_MAX       = 4095        # 12-bit
DAC_VOLTS     = 3.3         # V full scale
DAC_TARGET    = 2047        # 50% = 1.65V
DAC_TO_V      = DAC_VOLTS / DAC_MAX

# ── Ramp + noise onset ────────────────────────────────────────────────────
RAMP_RISE_MS  = 250.0       # ms linear rise before plateau / noise

# ── Noise ─────────────────────────────────────────────────────────────────
NOISE_AMP     = 1800        # ± DAC counts around DAC_TARGET
NOISE_SEED    = 12345       # fixed seed — same noise every run

# ── Chirp ─────────────────────────────────────────────────────────────────
CHIRP_F0      = 0.5         # Hz start
CHIRP_F1      = 200.0       # Hz end

# ── Sinusoid frequencies (Hz) ─────────────────────────────────────────────
SINE_FREQS    = [2, 10, 25, 60, 100, 500]

# ── Noise variants (name: lowpass cutoff Hz) ──────────────────────────────
NOISE_TYPES   = {'LP100': 100.0, 'LP500': 500.0}

# ── Repetitions per stimulus ──────────────────────────────────────────────
# Edit each line independently. Set to 0 to exclude a stimulus entirely.
REPS = {
    # Chirps
    'ChirpFwd':   20,
    'ChirpRev':   20,
    # Noise variants — keys must match NOISE_TYPES above
    'NoiseLp100': 20,
    'NoiseLp500': 20,
    # Sinusoids — keys must match f'Sine{f}Hz' for f in SINE_FREQS
    'Sine2Hz':    20,
    'Sine10Hz':   20,
    'Sine25Hz':   20,
    'Sine60Hz':   20,
    'Sine100Hz':  20,
    'Sine500Hz':  20,
    # Ramp
    'Ramp':       20,
}

# ── Session timing ────────────────────────────────────────────────────────
ISI_BASE_SEC   = 10.0       # s base inter-stimulus interval
ISI_JITTER_PCT = 20.0       # ± % jitter
RANDOM_SEED    = 12345      # fixed — same trial order every run

# ── Output directory for generated .h files ───────────────────────────────
OUTPUT_DIR = '.'

# ── Derived — do not edit ─────────────────────────────────────────────────
N             = int(STIM_DURATION * SAMPLE_RATE)
t             = np.linspace(0, STIM_DURATION, N, endpoint=False)
RAMP_RISE_N   = int(RAMP_RISE_MS / 1000 * SAMPLE_RATE)
NOISE_TBL_LEN = SAMPLE_RATE

# Build the flat ordered stimulus list from REPS (excludes zeros)
# This is the canonical order used by both the session overview and
# the trial_sequence.h generator — everything derives from this.
STIM_ORDER = [name for name, reps in REPS.items() if reps > 0]
total_trials = sum(REPS[n] for n in STIM_ORDER)
est_min = total_trials * (STIM_DURATION + ISI_BASE_SEC) / 60

print(f'Active stimuli      : {len(STIM_ORDER)}')
print(f'Total trials        : {total_trials}')
print(f'Estimated duration  : ~{est_min:.1f} min')
print()
print(f'{"Stimulus":<14} {"Reps":>5}')
print('-' * 22)
for name in STIM_ORDER:
    print(f'{name:<14} {REPS[name]:>5}')
print('-' * 22)
print(f'{"TOTAL":<14} {total_trials:>5}')


## 2. Waveform generators

Run this cell to generate all waveforms in memory before plotting.


In [ ]:
def dac2v(x): return np.asarray(x, float) * DAC_TO_V

def make_chirp():
    k = (CHIRP_F1/CHIRP_F0)**(1/STIM_DURATION)
    phase = 2*np.pi*CHIRP_F0*(k**t - 1)/np.log(k)
    return (0.5 + 0.5*np.sin(phase)) * DAC_TARGET

def make_ramp():
    sig = np.full(N, float(DAC_TARGET))
    sig[:RAMP_RISE_N] = np.linspace(0, DAC_TARGET, RAMP_RISE_N)
    return sig

def make_noise_lp(cutoff_hz, seed):
    """Lowpass FIR-filtered white noise, ramp onset, centred at DAC_TARGET."""
    rng  = np.random.default_rng(seed)
    raw  = rng.uniform(-1.0, 1.0, size=N + 2000)
    nyq  = SAMPLE_RATE / 2.0
    taps = firwin(1001, cutoff_hz/nyq, pass_zero=True, window='hamming')
    filt = lfilter(taps, 1.0, raw)[1001:][:N]
    filt = filt / np.max(np.abs(filt)) * NOISE_AMP
    noise_part = np.clip(DAC_TARGET + filt, 0, DAC_MAX)
    # Ramp onset: same 250ms rise as Ramp stimulus
    sig = noise_part.copy()
    ramp_up = np.linspace(0, 1, RAMP_RISE_N)
    sig[:RAMP_RISE_N] = noise_part[:RAMP_RISE_N] * ramp_up
    return sig

def make_sine(freq):
    return (0.5 + 0.5*np.sin(2*np.pi*freq*t)) * DAC_TARGET

# Generate all waveforms
chirp_fwd = make_chirp()
chirp_rev = chirp_fwd[::-1].copy()
ramp      = make_ramp()
noises    = {name: make_noise_lp(cutoff, NOISE_SEED ^ (i*0xABCD1234))
             for i, (name, cutoff) in enumerate(NOISE_TYPES.items())}
sines     = {f'Sine{f}Hz': make_sine(f) for f in SINE_FREQS}

all_waveforms = {
    'ChirpFwd': chirp_fwd, 'ChirpRev': chirp_rev,
    **{f'Noise{k}': v for k,v in noises.items()},
    **sines,
    'Ramp': ramp,
}

COLORS = {
    'ChirpFwd':'#534AB7','ChirpRev':'#0F6E56',
    'NoiseLp100':'#993C1D','NoiseLp500':'#712B13',
    'Ramp':'#854F0B',
}
# Sine colors: gradient of blues
sine_cols = plt.cm.Blues(np.linspace(0.35, 0.9, len(SINE_FREQS)))
for i, f in enumerate(SINE_FREQS):
    COLORS[f'Sine{f}Hz'] = sine_cols[i]

print('Waveforms generated:')
print(f'{"Name":<16} {"Min(V)":>7} {"Max(V)":>7} {"Mean(V)":>8}')
print('-'*42)
for name, sig in all_waveforms.items():
    v = dac2v(sig)
    print(f'{name:<16} {v.min():>7.3f} {v.max():>7.3f} {v.mean():>8.3f}')

## 3. Generate Arduino header files

This cell produces two `.h` files to copy into your Arduino sketch folder:
- **`noise_tables.h`** — LP100 and LP500 noise tables in PROGMEM flash
- **`trial_sequence.h`** — the pre-shuffled trial order in PROGMEM flash

The noise tables written here are **exactly** the waveforms you inspect in Section 4.
The trial sequence written here is **exactly** what the Arduino will execute.


In [ ]:
def arr_to_c_progmem(arr, name, dtype='int16_t', cols=16):
    lines = [f'const {dtype} {name}[{len(arr)}] PROGMEM = {{']
    row = []
    for i, v in enumerate(arr):
        row.append(f'{int(v):6d}')
        if len(row) == cols or i == len(arr)-1:
            sep = ',' if i < len(arr)-1 else ''
            lines.append('  ' + ', '.join(row) + sep)
            row = []
    lines.append('};')
    return '\n'.join(lines)

# ── noise_tables.h ────────────────────────────────────────────────────────
def noise_int16(cutoff_hz, seed):
    rng  = np.random.default_rng(seed)
    raw  = rng.uniform(-1.0, 1.0, size=NOISE_TBL_LEN + 2000)
    nyq  = SAMPLE_RATE / 2.0
    taps = firwin(1001, cutoff_hz/nyq, pass_zero=True, window='hamming')
    filt = lfilter(taps, 1.0, raw)[1001:][:NOISE_TBL_LEN]
    filt = filt / np.max(np.abs(filt)) * NOISE_AMP
    return np.round(filt).astype(np.int16)

lp100_tbl = noise_int16(100.0, NOISE_SEED)
lp500_tbl = noise_int16(500.0, NOISE_SEED ^ 0xABCD1234)

noise_h  = '// noise_tables.h — Auto-generated by inspect_stimuli_v3.ipynb\n'
noise_h += '// DO NOT EDIT — re-run notebook to regenerate\n'
noise_h += f'// LP100: lowpass 0-100Hz   LP500: lowpass 0-500Hz\n'
noise_h += f'// {NOISE_TBL_LEN} samples each (1s at {SAMPLE_RATE}Hz), looped\n'
noise_h += f'// Amplitude: int16 +-{NOISE_AMP} around DAC_TARGET={DAC_TARGET}\n\n'
noise_h += '#pragma once\n#include <avr/pgmspace.h>\n\n'
noise_h += arr_to_c_progmem(lp100_tbl, 'NOISE_LP100_TBL') + '\n\n'
noise_h += arr_to_c_progmem(lp500_tbl, 'NOISE_LP500_TBL') + '\n'

path_noise = os.path.join(OUTPUT_DIR, 'noise_tables.h')
with open(path_noise, 'w') as f: f.write(noise_h)
print(f'Written: {path_noise}  ({os.path.getsize(path_noise)//1024} KB)')

# ── trial_sequence.h ──────────────────────────────────────────────────────
# Enum mapping — must match StimulusType in opto_stim_v6.ino
STIM_ENUM = {
    'ChirpFwd':0,'ChirpRev':1,'NoiseLp100':2,'NoiseLp500':3,
    'Sine2Hz':4,'Sine10Hz':5,'Sine25Hz':6,'Sine60Hz':7,
    'Sine100Hz':8,'Sine500Hz':9,'Ramp':10
}

# Build trial list: REPS[name] copies of each active stimulus
trial_names = []
for name in STIM_ORDER:
    trial_names += [name] * REPS[name]

# Shuffle with fixed seed — same order every run
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(trial_names)
trial_codes = [STIM_ENUM[n] for n in trial_names]

seq_h  = '// trial_sequence.h — Auto-generated by inspect_stimuli_v3.ipynb\n'
seq_h += '// DO NOT EDIT — re-run notebook to regenerate\n'
seq_h += f'// {len(trial_codes)} trials, seed={RANDOM_SEED}\n'
seq_h += '// Active stimuli and reps:\n'
for name in STIM_ORDER:
    seq_h += f'//   {name}: {REPS[name]} reps\n'
seq_h += '// Enum codes: 0=ChirpFwd 1=ChirpRev 2=NoiseLp100 3=NoiseLp500\n'
seq_h += '//             4=Sine2 5=Sine10 6=Sine25 7=Sine60 8=Sine100 9=Sine500 10=Ramp\n\n'
seq_h += '#pragma once\n#include <avr/pgmspace.h>\n\n'
seq_h += f'const int TOTAL_TRIALS = {len(trial_codes)};\n\n'
seq_h += arr_to_c_progmem(np.array(trial_codes, dtype=np.uint8),
                           'TRIAL_SEQUENCE', dtype='uint8_t') + '\n'

path_seq = os.path.join(OUTPUT_DIR, 'trial_sequence.h')
with open(path_seq, 'w') as f: f.write(seq_h)
print(f'Written: {path_seq}  ({os.path.getsize(path_seq)} bytes)')
print(f'\nFirst 15 trials: {trial_names[:15]}')


## 4. Session overview

Full trial sequence visualised as a timeline — exactly what the Arduino will execute.
This is the authoritative record of your session structure.


In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle(
    f'Session overview — {len(trial_codes)} trials  ~{est_min:.0f} min  seed={RANDOM_SEED}',
    fontsize=12, y=0.98)

gs = gridspec.GridSpec(3, 2, figure=fig,
                       left=0.08, right=0.97,
                       top=0.93, bottom=0.07,
                       hspace=0.55, wspace=0.35,
                       height_ratios=[1.4, 1, 1])

n_types  = len(stim_labels)
stim_col = [COLORS.get(s, '#888') for s in stim_labels]

# Simulate jittered ISIs for the overview (same seed each run)
rng2     = np.random.default_rng(42)
isis     = ISI_BASE_SEC * (1 + rng2.uniform(
               -ISI_JITTER_PCT/100, ISI_JITTER_PCT/100, len(trial_codes)))
onsets_s = np.cumsum(isis) + np.arange(len(trial_codes)) * STIM_DURATION
onsets_m = onsets_s / 60

# ── 1. Dot raster ─────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, :])
for onset, code_idx in zip(onsets_m, trial_codes):
    ax.plot(onset, code_idx, marker='|', markersize=8, markeredgewidth=1.5,
            color=stim_col[code_idx], alpha=0.7)
ax.set_yticks(range(n_types))
ax.set_yticklabels(stim_labels, fontsize=8.5)
ax.set_xlabel('Time (min)', fontsize=10)
ax.set_xlim(0, onsets_m[-1] + 1)
ax.set_ylim(-0.6, n_types - 0.4)
ax.set_title('Trial onset raster — each tick = one trial', fontsize=10)
ax.grid(axis='x', lw=0.4, alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# ── 2. Cumulative trial count ──────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
for code_idx, name in enumerate(stim_labels):
    mask   = np.array(trial_codes) == code_idx
    times  = np.sort(onsets_m[mask])
    xs = np.concatenate([[0], np.repeat(times, 2), [onsets_m[-1] + 1]])
    ys = np.concatenate([[0], np.arange(1, len(times)+1).repeat(2), [len(times)]])
    ax.plot(xs, ys, color=COLORS.get(name, '#888'), lw=1.2, alpha=0.85)
ax.set_xlabel('Time (min)', fontsize=10)
ax.set_ylabel('Cumulative trials', fontsize=10)
ax.set_title('Cumulative trial count per stimulus\n(parallel lines = even spacing)', fontsize=10)
ax.set_xlim(0, onsets_m[-1] + 1)
ax.set_ylim(0, max(REPS.values()) + 1)
ax.grid(lw=0.4, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# ── 3. Inter-trial interval boxplots ──────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
itd_data = []
for code_idx, name in enumerate(stim_labels):
    mask  = np.array(trial_codes) == code_idx
    times = np.sort(onsets_m[mask])
    if len(times) > 1:
        itd_data.append((name, np.diff(times), COLORS.get(name, '#888')))
for pos, (name, itds, col) in enumerate(itd_data):
    ax.boxplot(itds, positions=[pos], vert=False, widths=0.5,
               patch_artist=True,
               boxprops=dict(facecolor=col, alpha=0.6),
               medianprops=dict(color='white', lw=1.5),
               whiskerprops=dict(color=col),
               capprops=dict(color=col),
               flierprops=dict(marker='.', color=col, markersize=3, alpha=0.5))
ax.set_yticks(range(len(itd_data)))
ax.set_yticklabels([x[0] for x in itd_data], fontsize=8.5)
ax.set_xlabel('Time between same-type trials (min)', fontsize=10)
ax.set_title('Inter-trial interval per stimulus\n(spread = good randomisation)', fontsize=10)
ax.grid(axis='x', lw=0.4, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# ── 4. Trial sequence strip ────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, :])
for i, code_idx in enumerate(trial_codes):
    ax.bar(i, 1, color=stim_col[code_idx], width=1.0, linewidth=0)
ax.set_xlim(0, len(trial_codes))
ax.set_xlabel('Trial number', fontsize=10)
ax.set_yticks([])
ax.set_title('Trial sequence — each column = one trial', fontsize=10)
ax.spines[['top', 'right', 'left']].set_visible(False)
patches = [mpatches.Patch(color=COLORS.get(s,'#888'), label=s) for s in stim_labels]
ax.legend(handles=patches, ncol=6, fontsize=8,
          loc='upper center', bbox_to_anchor=(0.5, -0.45), frameon=False)

plt.savefig('session_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: session_overview.png')


## 5. Individual stimulus inspection

Change `STIM_TO_INSPECT` to any key from `all_waveforms` to look at a specific stimulus.
The loop at the bottom plots all stimuli.


In [ ]:
def plot_one(name, sig, t_zoom=None, xlim_psd=(0,600)):
    col = COLORS.get(name, '#555')
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    fig.suptitle(name, color=col, fontweight='bold', x=0.01, ha='left', fontsize=12)

    ax = axes[0]
    thin = max(1, N//5000)
    ax.plot(t[::thin], dac2v(sig[::thin]), color=col, lw=0.7)
    ax.axhline(DAC_TARGET*DAC_TO_V, color='#ccc', lw=0.6, ls='--',
               label=f'{DAC_TARGET*DAC_TO_V:.2f}V centre')
    ax.set_xlim(0, STIM_DURATION); ax.set_ylim(-0.1, DAC_VOLTS+0.1)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Voltage (V)')
    ax.set_title('Waveform'); ax.legend(fontsize=8)
    if t_zoom:
        i0,i1 = int(t_zoom[0]*SAMPLE_RATE), int(t_zoom[1]*SAMPLE_RATE)
        ins = ax.inset_axes([0.55,0.08,0.42,0.48])
        ins.plot(t[i0:i1]*1000, dac2v(sig[i0:i1]), color=col, lw=0.9)
        ins.set_xlabel('ms',fontsize=7); ins.tick_params(labelsize=7)
        ins.spines[['top','right']].set_visible(False)
        ax.indicate_inset_zoom(ins, edgecolor='#bbb', lw=0.5)

    ax = axes[1]
    fp, pxx = welch(dac2v(sig), fs=SAMPLE_RATE,
                    nperseg=min(4096,N//4), scaling='density')
    ax.semilogy(fp, pxx, color=col, lw=1.0)
    ax.set_xlim(*xlim_psd)
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (V²/Hz)')
    ax.set_title('Power spectral density')
    plt.tight_layout(rect=[0,0,1,0.92])
    plt.show()

# Plot all — comment out any you don't need
plot_one('ChirpFwd',    chirp_fwd,          t_zoom=(0,0.3),    xlim_psd=(0,250))
plot_one('ChirpRev',    chirp_rev,          t_zoom=(4.7,5.0),  xlim_psd=(0,250))
plot_one('NoiseLP100',  noises['LP100'],    t_zoom=(0,0.05))
plot_one('NoiseLP500',  noises['LP500'],    t_zoom=(0,0.02))
plot_one('Ramp',        ramp,               t_zoom=(0,0.4))
for f in SINE_FREQS:
    zoom = min(2/f, 0.5)  # show ~2 cycles
    xlim = (0, min(f*3, 600))
    plot_one(f'Sine{f}Hz', sines[f'Sine{f}Hz'],
             t_zoom=(0, zoom), xlim_psd=xlim)

## 6. Noise deep-dive

Compares LP100 and LP500 side-by-side: PSD, amplitude distribution, clipping check.
The ramp onset is visible in the waveform for the first 250ms.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('Noise deep-dive', fontweight='bold', x=0.01, ha='left', fontsize=13)

for row, (nname, cutoff) in enumerate(NOISE_TYPES.items()):
    key = f'Noise{nname}'
    sig = noises[nname]
    col = COLORS.get(key, '#993C1D')

    # PSD
    ax = axes[row,0]
    fp, pxx = welch(dac2v(sig), fs=SAMPLE_RATE, nperseg=4096)
    ax.semilogy(fp, pxx, color=col, lw=1.0)
    ax.axvline(cutoff, color=col, lw=0.8, ls=':', label=f'cutoff {cutoff:.0f}Hz')
    ax.axvspan(0, cutoff, alpha=0.08, color=col)
    in_b  = pxx[(fp > 0) & (fp <= cutoff)]
    out_b = pxx[fp > cutoff]
    snr = 10*np.log10(np.mean(in_b)/np.mean(out_b)) if len(out_b) else np.inf
    cv  = np.std(in_b)/np.mean(in_b)*100
    ax.set_xlim(0, min(cutoff*3, 600))
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (V²/Hz)')
    ax.set_title(f'{nname}  SNR={snr:.1f}dB  CV={cv:.1f}%')
    ax.legend(fontsize=8)

    # Distribution
    ax = axes[row,1]
    v = dac2v(sig)
    ax.hist(v, bins=80, color=col, alpha=0.75, edgecolor='none')
    ax.axvline(DAC_TARGET*DAC_TO_V, color='#444', lw=1.0, ls='--',
               label=f'centre {DAC_TARGET*DAC_TO_V:.2f}V')
    ax.set_xlabel('Voltage (V)'); ax.set_ylabel('Count')
    ax.set_title(f'{nname}  mean={v.mean():.3f}V  std={v.std():.3f}V')
    ax.legend(fontsize=8)

    # Clipping + ramp onset
    ax = axes[row,2]
    thin = max(1, N//5000)
    ax.plot(t[::thin], dac2v(sig[::thin]), color=col, lw=0.5)
    ax.axhline(DAC_VOLTS, color='red', lw=0.8, ls='--', label='clip ceiling')
    ax.axhline(0,         color='red', lw=0.8, ls='--', label='clip floor')
    ax.axhline(DAC_TARGET*DAC_TO_V, color='#aaa', lw=0.6, ls=':', label='centre')
    ax.axvline(RAMP_RISE_MS/1000, color='#888', lw=0.8, ls=':',
               label=f'{RAMP_RISE_MS:.0f}ms ramp end')
    n_clip = int(np.sum((sig<=0)|(sig>=DAC_MAX)))
    ax.set_ylim(-0.2, DAC_VOLTS+0.2)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Voltage (V)')
    ax.set_title(f'{nname}  clipped={n_clip}/{N}')
    ax.legend(fontsize=7)

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

## 7. Chirp deep-dive

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Chirp deep-dive', fontweight='bold', x=0.01, ha='left', fontsize=13)

for row, (name, sig, fwd) in enumerate([
        ('ChirpFwd', chirp_fwd, True),
        ('ChirpRev', chirp_rev, False)]):
    col = COLORS[name]
    k   = (CHIRP_F1/CHIRP_F0)**(1/STIM_DURATION)
    freq = CHIRP_F0 * k**t
    if not fwd: freq = freq[::-1]

    ax = axes[row,0]
    ax.plot(t, freq, color=col, lw=1.0)
    ax.set_yscale('log'); ax.set_yticks([1,10,100])
    ax.set_yticklabels(['1','10','100'])
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Inst. freq (Hz)')
    ax.set_title(f'{name} — log frequency sweep')

    ax = axes[row,1]
    fs, ts, Sxx = spectrogram(dac2v(sig), fs=SAMPLE_RATE, nperseg=512, noverlap=480)
    im = ax.pcolormesh(ts, fs, 10*np.log10(Sxx+1e-12), cmap='magma', shading='gouraud')
    ax.set_ylim(0, 210)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Freq (Hz)')
    ax.set_title(f'{name} — spectrogram')
    plt.colorbar(im, ax=ax, label='dB', pad=0.02)

    ax = axes[row,2]
    # High-freq end for Fwd = last 50ms; for Rev = first 50ms
    i0,i1 = (N-500,N) if fwd else (0,500)
    lbl = 'last 50ms (200Hz end)' if fwd else 'first 50ms (200Hz end)'
    ax.plot(np.arange(i1-i0)/SAMPLE_RATE*1000, dac2v(sig[i0:i1]),
            color=col, lw=1.0)
    ax.set_xlabel('ms'); ax.set_ylabel('V')
    ax.set_title(f'{name} — {lbl}')

plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

## 8. Sinusoid overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('Sinusoid stimuli — all 6 frequencies', fontsize=11)
for ax, f in zip(axes.flat, SINE_FREQS):
    sig = sines[f'Sine{f}Hz']
    col = COLORS[f'Sine{f}Hz']
    # show 3 cycles or 500ms, whichever shorter
    show_s = min(3/f, 0.5)
    n_show = int(show_s * SAMPLE_RATE)
    ax.plot(t[:n_show]*1000, dac2v(sig[:n_show]), color=col, lw=1.0)
    ax.set_ylim(-0.1, DAC_VOLTS+0.1)
    ax.set_xlabel('ms'); ax.set_ylabel('V')
    samp_per_cycle = SAMPLE_RATE/f
    ax.set_title(f'{f} Hz  ({samp_per_cycle:.0f} samples/cycle)')
plt.tight_layout()
plt.show()

# Flag if any frequency has fewer than 10 samples/cycle
for f in SINE_FREQS:
    spc = SAMPLE_RATE/f
    flag = ' *** LOW RESOLUTION' if spc < 20 else ''
    print(f'Sine {f:>5} Hz : {spc:6.1f} samples/cycle{flag}')

## 9. Save all stimulus figures

In [ ]:
os.makedirs('figures', exist_ok=True)
for name, sig in all_waveforms.items():
    col = COLORS.get(name,'#555')
    fig, axes = plt.subplots(1,2,figsize=(11,3))
    thin = max(1,N//5000)
    axes[0].plot(t[::thin], dac2v(sig[::thin]), color=col, lw=0.7)
    axes[0].axhline(DAC_TARGET*DAC_TO_V, color='#ccc', lw=0.5, ls='--')
    axes[0].set_ylim(-0.1,DAC_VOLTS+0.1)
    axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('V')
    axes[0].set_title('Waveform')
    fp,pxx = welch(dac2v(sig),fs=SAMPLE_RATE,nperseg=min(4096,N//4))
    axes[1].semilogy(fp,pxx,color=col,lw=1.0)
    axes[1].set_xlim(0,600); axes[1].set_xlabel('Hz')
    axes[1].set_ylabel('PSD'); axes[1].set_title('PSD')
    fig.suptitle(name,color=col,fontweight='bold',x=0.01,ha='left')
    plt.tight_layout(rect=[0,0,1,0.93])
    plt.savefig(f'figures/{name}.png',dpi=150,bbox_inches='tight')
    plt.close()
    print(f'Saved figures/{name}.png')